# Scraping from Lovecrafts

### Scraping every product link

In [21]:
import os
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

target_dir = "../frontend/src/assets/images"
os.makedirs(target_dir, exist_ok=True)

def scrape_product(product_link, image_dir="../frontend/src/assets/images"):
    row = {}
    os.makedirs(image_dir, exist_ok=True)
    
    try:
        r = requests.get(product_link, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")
    except Exception as e:
        print(f"Connection error for {product_link}: {e}")
        return row
    
    title_element = soup.find("h1", class_="sf-heading__title title")
    if title_element:
        raw_title = title_element.get_text(strip=True)
        row["title"] = raw_title.strip()
    else:
        row["title"] = ""

    names = soup.find_all("dt", class_="sf-property__name")
    values = soup.find_all("dd", class_="sf-property__value")

    for name, value in zip(names, values):
        key = name.get_text(strip=True).lower().replace(":", "").replace(" ", "_")
        row[key] = value.get_text(strip=True)
    row["merged_info"] = " ".join(p.get_text(strip=True) for p in values)

    desc_container = soup.find("div", {"data-testid": "product-description"})
    if desc_container:
        row["description"] = desc_container.get_text(" ", strip=True)
    else:
        row["description"] = ""

    # Image Extraction and Download
    img_container = soup.find("span", class_="sf-image--wrapper sf-gallery__big-image")
    img_tag = img_container.find("img") if img_container else None
    
    img_url = None
    if img_tag:
        img_url = img_tag.get("src")
    
    row['image_url'] = img_url

    if img_url:
        safe_filename = re.sub(r'[^\w\s-]', '', row["title"]).strip().replace(' ', '_')
        save_path = os.path.join(image_dir, f"{safe_filename}.jpg")

        try:
            img_res = requests.get(img_url, headers=headers, stream=True, timeout=10)
            if img_res.status_code == 200 and 'image' in img_res.headers.get('Content-Type', ''):
                with open(save_path, 'wb') as f:
                    for chunk in img_res.iter_content(1024):
                        f.write(chunk)
                row["local_path"] = save_path
            else:
                print(f"Invalid image response for: {row['title']}")
        except Exception as e:
            print(f"Error downloading image for {row['title']}: {e}")
    return row


In [ ]:
base_url = "https://www.lovecrafts.com/en-us/l/crochet/crochet-patterns/free-crochet-patterns"
rows = []
page = 1
max_pages = 100

while page <= max_pages:
    #print(f"Scraping page {page}...")
    url = base_url if page == 1 else f"{base_url}?page={page}"
    #print(url)

    response = requests.get(url, headers=headers)
    #print(len(response.text))
    soup = BeautifulSoup(response.text, "html.parser")

    cards = soup.find_all("div", {"class": "lc-product-card"})
    #print (len(cards))
    
    if not cards:
        print("No more products found. Ending scrape.")
        break

    for card in cards:
        link_tag = card.find("a", class_="lc-product-card__link")
        
        if link_tag:
            href = link_tag.get("href")
            if not href.startswith("http"):
                full_url = f"https://www.lovecrafts.com{href}"
            else:
                full_url = href
                
            try:
                row = scrape_product(full_url, image_dir=target_dir)
                row["product_link"] = full_url
                rows.append(row)
            except Exception as e:
                print(f"Error scraping {full_url}: {e}")
            
            time.sleep(0.5)

    page += 1

In [25]:
df = pd.DataFrame(rows)
df.columns = [col.replace(':', '') for col in df.columns]
print(f"Scraping complete! Scraped {len(df)} products.")

Scraping complete! Scraped 1850 products.


In [26]:
df.head()

,title,brand,craft,designer,format,language,notes,number_of_patterns,pages,skill_level,...,image_url,local_path,product_link,featured_product,crochet_hooks,finished_size,techniques_and_construction,needles_required,crochet_thread_size,craft_category
0,Jellyfish Babies,Independent Designer,Crochet,Jade Gauthier-Boutin,Downloadable PDF,"English, French",• Two 6 mm safety eyes,1,14,Beginner,...,https://isv.prod.lovecrafts.co/v1/images/87236...,../frontend/src/assets/images/Jellyfish_Babies...,https://www.lovecrafts.com/en-us/p/jellyfish-b...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Sakura Market Bag,Independent Designer,Crochet,K.A.M.E. Crochet,Downloadable PDF,"Dutch, English, French, German",NaN,1,33,Intermediate,...,https://isv.prod.lovecrafts.co/v1/images/cd77c...,../frontend/src/assets/images/Sakura_Market_Ba...,https://www.lovecrafts.com/en-us/p/sakura-mark...,Paintbox Yarns Cotton DK,NaN,NaN,NaN,NaN,NaN,NaN
2,Easy Peasy Crochet Baby Blanket in Caron One P...,Caron,Crochet,NaN,Downloadable PDF,English,NaN,1,2,Beginner,...,https://isv.prod.lovecrafts.co/v1/images/8a9cf...,../frontend/src/assets/images/Easy_Peasy_Croch...,https://www.lovecrafts.com/en-us/p/easy-peasy-...,Caron One Pound,6.00mm (J),Blanket: 40in square,NaN,NaN,NaN,NaN
3,Baby Turtles,Independent Designer,Crochet,Jade Gauthier-Boutin,Downloadable PDF,"English, French","• Two 6 mm safety eyes or buttons, beads, felt...",1,12,Beginner,...,https://isv.prod.lovecrafts.co/v1/images/c5cdb...,../frontend/src/assets/images/Baby_Turtles.jpg,https://www.lovecrafts.com/en-us/p/baby-turtle...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Squishy Baby Octopus,Independent Designer,Crochet,Jade Gauthier-Boutin,Downloadable PDF,"English, French",Two 12 mm safety eyes,1,6,Beginner,...,https://isv.prod.lovecrafts.co/v1/images/95d72...,../frontend/src/assets/images/Squishy_Baby_Oct...,https://www.lovecrafts.com/en-us/p/squishy-bab...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
df['local_path'] = df['local_path'].str.replace("../frontend/src/assets/images/", "", regex=False).str.strip()

In [29]:
df['skill_level'].value_counts()

skill_level
Beginner             864
Intermediate         523
Advanced              34
Advanced Beginner      6
Name: count, dtype: int64

In [32]:
df.loc[df['skill_level']== 'Advanced Beginner', 'skill_level'] = "Advanced"

In [33]:
df['skill_level'].value_counts()

skill_level
Beginner        864
Intermediate    523
Advanced         40
Name: count, dtype: int64

In [34]:
df.to_csv("../data/lovecrafts_crochet_patterns.csv", index=False)